# 5. Repensando el Overfitting

#### 5.1 Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/dm"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dm"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/itba2026-7c9a/dm/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"




---



## 5.2 rpart  Canaritos

Se agregarán canaritos al dataset, se entrenará el arbol con los mejores hiperparámetros encontrados, y se analizará si los canaritos aparecen en algun split.

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [1]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,668112,35.7,1473245,78.7,1425915,76.2
Vcells,1236259,9.5,8388608,64.0,1978677,15.1


In [2]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
if(!require("rpart.plot")) install.packages("rpart.plot")
require("rpart.plot")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: rpart

Loading required package: rpart.plot

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘rpart.plot’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Loading required package: rpart.plot



### 5.2.1  carga manual de hiperparámetros
Aqui debe cargar SU semilla primigenia y
<br> SUS mejores hiperparámetros que encontró para el ARBOL DE DECISION, ya sea por Grid Search o  Bayesian Optimization

In [3]:
PARAM <- list()
PARAM$semilla_primigenia <- 100019

PARAM$rpart$cp <- -0.5
PARAM$rpart$minsplit <- 500
PARAM$rpart$minbucket <- 100
PARAM$rpart$maxdepth <- 8

In [4]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp5200"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [5]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [6]:
# me quedo solo con los datos de julio
dataset <- dataset[ foto_mes==202107,]

In [7]:
# uso esta semilla para los canaritos
set.seed(PARAM$semila_primigenia)

# agrego los siguientes canaritos
for( i in 1:154 ) dataset[ , paste0("canarito", i ) :=  runif( nrow(dataset)) ]

la siguiente celda tarda 4 minutos en correr

In [8]:
# Entreno el modelo

modelo <- rpart(formula= "clase_ternaria ~ .",
  data= dataset,
  model= TRUE,
  xval= 0,
  control= PARAM$rpart
)


In [9]:
# genero un pdf con el dibujo del arbol

pdf(file = "arbol_canaritos.pdf", width=28, height=4)
prp(modelo, extra=101, digits=5, branch=1, type=4, varlen=0, faclen=0)
dev.off()

Warning message:
“labs do not fit even at cex 0.15, there may be some overplotting”


agg_record_539b643818f2 
                      2

vaya a su Google Drive
<br> busque la carpeta **My Drive / labo1 / exp / exp5200**
<br> baje el archivo **arbol_canaritos.pdf**  a su laptop
<br> abra el .pdf con el Acrobat Reader
<br> y dentro del .pdf busque splits hechos en alguna de las nuevas variables canaritos



---



## 5.3 rpart  Canaritos desconfiados

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [10]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,836527,44.7,1473245,78.7,1473245,78.7
Vcells,1649215,12.6,172348052,1315.0,215425084,1643.6


In [11]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
if(!require("rpart.plot")) install.packages("rpart.plot")
require("rpart.plot")

Aqui debe cargar SU semilla primigenia y
<br> parametros de un@ alumn@ desconfiad@

In [12]:
PARAM <- list()
PARAM$semilla_primigenia <- 102191

PARAM$rpart$cp <- -0.5
PARAM$rpart$minsplit <- 2000
PARAM$rpart$minbucket <- 800
PARAM$rpart$maxdepth <- 6

In [13]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp5300"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [14]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [15]:
# me quedo solo con los datos de julio
dataset <- dataset[ foto_mes==202107,]

In [16]:
# uso esta semilla para los canaritos
set.seed(PARAM$semila_primigenia)

# agrego los siguientes canaritos
for( i in 1:154 ) dataset[ , paste0("canarito", i ) :=  runif( nrow(dataset)) ]

la siguiente celda tarda 4 minutos en correr

In [17]:
# Entreno el modelo

modelo <- rpart(formula= "clase_ternaria ~ .",
  data= dataset,
  model= TRUE,
  xval= 0,
  control= PARAM$rpart
)


In [18]:
# genero un pdf con el dibujo del arbol

pdf(file = "arbol_canaritos_desconfiados.pdf", width=28, height=4)
prp(modelo, extra=101, digits=5, branch=1, type=4, varlen=0, faclen=0)
dev.off()

agg_record_539b25e9a316 
                      2

vaya a su Google Drive
<br> busque la carpeta **My Drive /  dm / labo1 / exp5300**
<br> baje el archivo **arbol_canaritos_desconfiados.pdf**  a su laptop
<br> abra el .pdf con el Acrobat Reader
<br> y dentro del .pdf busque splits hecho en alguna de las nuevas variables canaritos



---



## 5.4 rpart  Canaritos pruning

Se trabaja con la original clase_ternaria

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [19]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,836533,44.7,1473245,78.7,1473245,78.7
Vcells,1649225,12.6,165518130,1262.9,215425084,1643.6


In [20]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
if(!require("rpart.plot")) install.packages("rpart.plot")
require("rpart.plot")

Aqui debe cargar SU semilla primigenia

In [21]:
PARAM <- list()
PARAM$semilla_primigenia <- 100019

PARAM$peso <- 10

# Dejo crecer el arbol sin ninguna limitacion
# sin limite de altura ( 30 es el maximo que permite rpart )
# sin limite de minsplit ( 2 es el minimo natural )
# sin limite de minbukcet( 1 es el minimo natural )
# ya aprendimos que cp debe ser negativo
PARAM$rpart$cp <- -1
PARAM$rpart$minsplit <- 2
PARAM$rpart$minbucket <- 1
PARAM$rpart$maxdepth <- 16  # deberia pohner 31, por velocidad en la clase va el 16

In [22]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp5400"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [23]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [24]:
# uso esta semilla para los canaritos
set.seed(PARAM$semila_primigenia)

# agrego los siguientes canaritos
for( i in 1:155 ) dataset[ , paste0("canarito", i ) :=  runif( nrow(dataset)) ]

In [25]:
# datos de training
dtrain <- dataset[foto_mes == 202107]

la siguiente celda corre en 12 minutos

In [26]:
# Entreno el modelo
pesos <- dtrain[, ifelse( clase_ternaria=="BAJA+2", PARAM$peso, 1.0 ) ]

modelo_original <- rpart(formula= "clase_ternaria ~ .",
  data= dtrain,
  model= TRUE,
  xval= 0,
  control= PARAM$rpart,
  weights= pesos
)


In [27]:
# hago el pruning de los canaritos
# haciendo un hackeo a la estructura  modelo_original$frame
# -666 es un valor arbritrariamente negativo que jamas es generado por rpart

modelo_original$frame[
    modelo_original$frame$var %like% "canarito",
    "complexity"
] <- -666

modelo_pruned <- prune(modelo_original, -666)

In [28]:
# genero un pdf con el dibujo del arbol

pdf(file = "stopping_at_canaritos.pdf", width=28, height=4)
prp(modelo_pruned, extra=101, digits=5, branch=1, type=4, varlen=0, faclen=0)
dev.off()

agg_record_539bfc9f6b0 
                     2

In [29]:
# datos del futuro
dfuture <- dataset[foto_mes == 202109]

In [30]:
# scoring, aplico el modelo a los datos del futuro
prediccion <- predict(modelo_pruned,
  dfuture,
  type= "prob"
)

In [31]:
# tabla prediccion
tb_prediccion <- as.data.table(list(
  "numero_de_cliente" = dfuture$numero_de_cliente,
  "prob"=prediccion[, "BAJA+2"]
))

In [32]:
# Decison
PARAM$prob_corte <-  PARAM$peso/ ( PARAM$peso + 39)

tb_prediccion[ , Predicted := ifelse( prob< PARAM$prob_corte, 0L, 1L) ]

In [33]:
# archivo para kaggle
archivo_kaggle <- "K5400_001.csv"

fwrite( tb_prediccion[, list(numero_de_cliente, Predicted)],
  file= archivo_kaggle,
  sep= ","
)

In [34]:
# subida a Kaggle
comando <- "kaggle competitions submit"
competencia <- "-c data-mining-inicial-2026-a"
arch <- paste( "-f", archivo_kaggle)

In [35]:
mensaje <- paste0("-m 'cp=", PARAM$rpart$cp,
  "  minsplit=", PARAM$rpart$minsplit,
  "  minbucket=", PARAM$rpart$minbucket,
  "  maxdepth=", PARAM$rpart$maxdepth,
  "  peso=", PARAM$peso,
  "'"
 )

In [36]:
linea <- paste( comando, competencia, arch, mensaje)

# este es el comando que correria desde el prompt de Linux
linea

[1] "kaggle competitions submit -c data-mining-inicial-2026-a -f K5400_001.csv -m 'cp=-1  minsplit=2  minbucket=1  maxdepth=16  peso=10'"

In [37]:
# ejecuto el comando
salida <- system(linea, intern=TRUE)
cat(salida)

Successfully submitted to Data Mining, Inicial 2026 A

vaya a su Google Drive
<br> busque la carpeta **My Drive /  labo1 / exp / exp5400**
<br> baje el archivo **stopping_at_canaritos.pdf**  a su laptop
<br> abra el .pdf con el Acrobat Reader




---



## 5.5 rpart  Canaritos pruning BINARIA

Pasamos a trabajar con una clase  Binaria


*   POS = { BAJA+1, BAJA+2 }
*   NEG = { CONTINUA }




ahora la probabilidad que devuelve el modelo es de POS,
<br> ya no es la de BAJA+2,
<br> ya no puedo cortar por ella
<br> debo cortar por cnatidad de envios !

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [38]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,843025,45.1,1964198,104.9,1819568,97.2
Vcells,1663433,12.7,374441757,2856.8,371433859,2833.9


In [39]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
if(!require("rpart.plot")) install.packages("rpart.plot")
require("rpart.plot")

Aqui debe cargar SU semilla primigenia

In [40]:
PARAM <- list()
PARAM$semilla_primigenia <- 100019

PARAM$envios <- 9000
PARAM$peso <- 10

# Dejo crecer el arbol sin ninguna limitacion
# sin limite de altura ( 30 es el maximo que permite rpart )
# sin limite de minsplit ( 2 es el minimo natural )
# sin limite de minbukcet( 1 es el minimo natural )
# ya aprendimos que cp debe ser negativo
PARAM$rpart$cp <- -1
PARAM$rpart$minsplit <- 2
PARAM$rpart$minbucket <- 1
PARAM$rpart$maxdepth <- 16 # deberia ser 31, por velocidad en clsae se baja  16

In [41]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp5500"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [42]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [43]:
# uso esta semilla para los canaritos
set.seed(PARAM$semila_primigenia)

# agrego los siguientes canaritos
for( i in 1:155 ) dataset[ , paste0("canarito", i ) :=  runif( nrow(dataset)) ]

In [44]:
# datos de training
dtrain <- dataset[foto_mes == 202107]

In [45]:
# clase binaria
dtrain[, clase_binaria2 := ifelse( clase_ternaria=="CONTINUA", "NEG", "POS" ) ]
dtrain[, clase_ternaria := NULL ]

la siguiente celda corre en 11 minutos

In [46]:
# Entreno el modelo
pesos <- dtrain[, ifelse( clase_binaria2=="POS", PARAM$peso, 1.0 ) ]

modelo_original <- rpart(formula= "clase_binaria2 ~ .",
  data= dtrain,
  model= TRUE,
  xval= 0,
  control= PARAM$rpart,
  weights= pesos
)


In [47]:
# hago el pruning de los canaritos
# haciendo un hackeo a la estructura  modelo_original$frame
# -666 es un valor arbritrariamente negativo que jamas es generado por rpart

modelo_original$frame[
    modelo_original$frame$var %like% "canarito",
    "complexity"
] <- -666

modelo_pruned <- prune(modelo_original, -666)

In [48]:
# genero un pdf con el dibujo del arbol

pdf(file = "stopping_at_canaritos.pdf", width=28, height=4)
prp(modelo_pruned, extra=101, digits=5, branch=1, type=4, varlen=0, faclen=0)
dev.off()

agg_record_539b5b335524 
                      2

In [49]:
# datos del futuro
dfuture <- dataset[foto_mes == 202109]

In [50]:
# scoring, aplico el modelo a los datos del futuro
prediccion <- predict(modelo_pruned,
  dfuture,
  type= "prob"
)

In [51]:
# tabla prediccion
tb_prediccion <- as.data.table(list(
  "numero_de_cliente" = dfuture$numero_de_cliente,
  "prob"=prediccion[, "POS"]
))

In [52]:
# Decison
setorder( tb_prediccion, -prob )
tb_prediccion[ , Predicted := 0L ]
tb_prediccion[ seq(PARAM$envios), Predicted := 1L ]


In [53]:
# archivo para kaggle
archivo_kaggle <- "KBin5500_005.csv"

fwrite( tb_prediccion[, list(numero_de_cliente, Predicted)],
  file= archivo_kaggle,
  sep= ","
)

In [54]:
# subida a Kaggle
comando <- "kaggle competitions submit"
competencia <- "-c data-mining-inicial-2026-a"
arch <- paste( "-f", archivo_kaggle)

In [55]:
mensaje <- paste0("-m 'cp=", PARAM$rpart$cp,
  "  minsplit=", PARAM$rpart$minsplit,
  "  minbucket=", PARAM$rpart$minbucket,
  "  maxdepth=", PARAM$rpart$maxdepth,
  "  envios=", PARAM$envios,
  "  peso=", PARAM$peso,
  "'"
 )

In [56]:
linea <- paste( comando, competencia, arch, mensaje)

# este es el comando que correria desde el prompt de Linux
linea

[1] "kaggle competitions submit -c data-mining-inicial-2026-a -f KBin5500_005.csv -m 'cp=-1  minsplit=2  minbucket=1  maxdepth=16  envios=9000  peso=10'"

In [57]:
# ejecuto el comando
salida <- system(linea, intern=TRUE)
cat(salida)

Successfully submitted to Data Mining, Inicial 2026 A

vaya a su Google Drive
<br> busque la carpeta **My Drive /  labo1 / exp / exp5500**
<br> baje el archivo **stopping_at_canaritos.pdf**  a su laptop
<br> abra el .pdf con el Acrobat Reader




---

